# Mushroom Edibility Classification – Solution

**Short name (GitHub):** `MushEdib`  
**Lab source:** UCI *Mushroom* table (Audubon Society field-guide codes) + the attached Data Prep / EDA / RF / Evaluation brief.  
**Data:** `data/mushrooms.csv` (8,124 × 23 letter codes). Target `class`: **e** = edible, **p** = poisonous.  
**Companion files:** `MushEdib_Solution.ipynb`, `MushEdib_Reusable_Template.ipynb`, `MushEdib.py`, `MushEdib_Cheatsheet.docx`, `MushEdib_Project_Memo.docx`, `MushEdib_Strategy_Guide.docx`, `MushEdib_1Page_Summary_Report.docx`, `mushedib_flowchart.png`.

This notebook is the worked key. Use `MushEdib_Practice_Skeleton.ipynb` to practice first.

**This is not a foraging app.** A perfect score on this 1987 codebook does not mean the next cap you pick is safe.

You will:

1. Clean `?` in `stalk-root` → `u`, check duplicates.
2. Plot class balance, odor × class, cap-color × class, and a 12-feature factorize heatmap.
3. Drop zero-variance `veil-type`, LabelEncode every column, 80/20 split (`random_state=42`).
4. Fit `RandomForestClassifier(random_state=42)` and evaluate.
5. Compare alternates, run extra practice, twist simulation knobs.



## Inline cheat-sheet (keep this cell visible)

See also **`MushEdib_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Load | `pd.read_csv("data/mushrooms.csv")` |
| Missing in this table | literal `"?"`, almost only in `stalk-root` (2,480 rows) |
| Task replacement | `df["stalk-root"] = df["stalk-root"].replace("?", "u")` |
| Collision | official codebook already uses `u` = cup. Task still wants `"u"` for unknown |
| Duplicates | `df.duplicated().sum()` then `drop_duplicates()` |
| Zero-variance | `veil-type` is always `p` — drop before encoding |
| Factorize (EDA only) | `df[col], _ = pd.factorize(df[col])` — arbitrary integer codes |
| Top-12 heatmap | drop `class` *before* `.head(12)`; heatmap is 12×12, not 13×13 |
| Encode for trees | `LabelEncoder().fit_transform` per column |
| Encode for distance / LR | `pd.get_dummies` (one-hot) |
| Split | `train_test_split(X, y, test_size=0.2, random_state=42)` |
| RF | `RandomForestClassifier(random_state=42)` — defaults, 100 trees |
| Costly cell | actual **p**, predicted **e** (false edible) |
| Odor rule | majority class per odor code ≈ 98.5% on this table |
| Class map after LE | alphabetical → `e=0`, `p=1` |

**sklearn note.** Trees do not need scaling. LabelEncoder invents a fake order (`brown < buff < cinnamon`) that is fine for RF / DT and wrong for kNN / unpenalized linear models.



## Flowchart of the desired outcome

![MushEdib flow](mushedib_flowchart.png)

Clean first (unknown stalk-root, no dups). Look at odor before you fit anything. Drop `veil-type`. Freeze the 80/20 seed at 42. Score the model on the costly cell, not only accuracy. Then break it on purpose in the simulation section.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    recall_score, f1_score,
)
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# optional local helper
try:
    import MushEdib as me
except ImportError:
    me = None
print("ready")


## 1. Data preparation

Load `data/mushrooms.csv`. Display the first 5 rows. Call `.info()`. Count `"?"` in `stalk-root` and replace them with `"u"`. Count and drop exact duplicate rows. Print the cleaned shape and `class` counts.

Expected: 8,124 rows × 23 columns, 2,480 question marks, 0 duplicates, class split 4,208 edible / 3,916 poisonous.


In [ ]:
df = pd.read_csv("data/mushrooms.csv")
print("First 5 rows:")
print(df.head())
print("\nDataFrame info:")
df.info()
n_q = int((df["stalk-root"] == "?").sum())
print(f"\nMissing ('?') values in 'stalk-root': {n_q}")
df["stalk-root"] = df["stalk-root"].replace("?", "u")
assert (df["stalk-root"] == "?").sum() == 0
n_dups = int(df.duplicated().sum())
print(f"Number of duplicate rows: {n_dups}")
if n_dups:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicate rows removed.")
print(f"Cleaned dataset shape: {df.shape}")
print(df["class"].value_counts())


### Alternate — treat `?` as its own readable label, or impute the mode

The brief asks for `"u"`. Two other legal choices: a new token `"missing"` (no collision with cup=`u`), or the mode (`b` = bulbous). Trees can learn from an explicit missing level; imputing the mode invents a root that was never observed.


In [ ]:
raw = pd.read_csv("data/mushrooms.csv")
print("mode of raw stalk-root (including ?):", raw["stalk-root"].mode().iloc[0])
df_missing = raw.copy()
df_missing["stalk-root"] = df_missing["stalk-root"].replace("?", "missing")
df_mode = raw.copy()
mode_root = raw.loc[raw["stalk-root"] != "?", "stalk-root"].mode().iloc[0]
df_mode["stalk-root"] = df_mode["stalk-root"].replace("?", mode_root)
print("unknown-as-missing levels:", sorted(df_missing["stalk-root"].unique()))
print("mode-imputed levels:", sorted(df_mode["stalk-root"].unique()))
print("We keep df with '?' → 'u' for the rest of the notebook.")


## 2. Exploratory data analysis

Plot three countplots, then a temporary factorize encoding, then the **12 × 12** heatmap of the top features by |corr| with `class`.

1. Class balance.
2. Odor vs class (the key feature).
3. Cap-color vs class (weak on its own).
4. `pd.factorize` every column into integers.
5. Absolute correlation with `class`, drop the target, take `.head(12)`, heatmap **those 12 only**.

Reference images (if you want a visual target): `mushedib_class_balance.png`, `mushedib_odor.png`, `mushedib_capcolor.png`, `mushedib_heatmap.png`.


In [ ]:
fig, ax = plt.subplots()
ax = sns.countplot(data=df, x="class", hue="class",
                   palette={"e": "#6aa84f", "p": "#cc4125"}, legend=False, ax=ax)
ax.set_title("Class balance: edible vs poisonous")
ax.set_xlabel("Class (e = edible, p = poisonous)")
total = len(df)
for container in ax.containers:
    labels = [f"{int(v.get_height())}\n({v.get_height()/total:.1%})" for v in container]
    ax.bar_label(container, labels=labels, padding=3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, x="odor", hue="class",
              palette={"e": "#6aa84f", "p": "#cc4125"}, ax=ax)
ax.set_title("Edibility by odor (key feature)")
ax.set_xlabel("odor  a=almond l=anise c=creosote y=fishy f=foul m=musty n=none p=pungent s=spicy")
plt.tight_layout(); plt.show()

order = df["cap-color"].value_counts().index
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, x="cap-color", hue="class", order=order,
              palette={"e": "#6aa84f", "p": "#cc4125"}, ax=ax)
ax.set_title("Edibility by cap color")
ax.set_xlabel("cap-color  n=brown b=buff c=cinnamon g=gray r=green p=pink u=purple e=red w=white y=yellow")
plt.tight_layout(); plt.show()

df_encoded = df.copy()
for col in df_encoded.columns:
    df_encoded[col], _ = pd.factorize(df_encoded[col])
print("Encoded preview:")
print(df_encoded.head())

correlations = df_encoded.corr()["class"].abs().drop("class")
top12 = correlations.sort_values(ascending=False).head(12)
print("\nTop 12 features correlated with class:")
print(top12)

top_features = list(top12.index)  # exactly 12 — do not append class
corr_matrix = df_encoded[top_features].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="Purples",
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation heatmap of top 12 features")
plt.tight_layout(); plt.show()


### What you should see

- Balance is close: **51.8% edible / 48.2% poisonous**. Accuracy is a usable headline, but the *costly* error is still a false edible.
- Odor nearly partitions the table. Almond (`a`) and anise (`l`) are edible; foul / fishy / spicy / pungent / creosote / musty are poisonous. `n` (none) is mixed — that is where the other columns earn their keep.
- Cap color overlaps heavily. Brown / yellow / white appear in both classes.
- Factorize |corr| ranking is indicative, not gospel (the codes are unordered). Typical leaders: **odor, spore-print-color, ring-type, stalk-surface-*, gill-size, bruises**.


## 3. Preprocessing

Drop `veil-type`. LabelEncode **every remaining column including `class`**. Split `X` / `y`. `train_test_split(..., test_size=0.2, random_state=42)`. Print the four shapes.

Expected shapes: `X_train (6499, 21)`, `X_test (1625, 21)`.


In [ ]:
df_model = df.drop(columns=["veil-type"])
print(f"Dropped 'veil-type'. Remaining columns: {len(df_model.columns)}")

label_encoders = {}
df_le = df_model.copy()
for col in df_le.columns:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col])
    label_encoders[col] = le
print("Encoded target classes:", dict(zip(
    label_encoders["class"].classes_,
    label_encoders["class"].transform(label_encoders["class"].classes_),
)))

X = df_le.drop(columns=["class"])
y = df_le["class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("test poisonous rate:", float(y_test.mean()))


### Alternate — one-hot instead of LabelEncoder

Keep this for logistic regression / kNN later. Do not replace `X_train` used by the forest.


In [ ]:
X_oh = pd.get_dummies(df.drop(columns=["veil-type", "class"]), drop_first=False)
y_oh = (df["class"] == "p").astype(int)
print("one-hot width:", X_oh.shape[1])
Xoh_train, Xoh_test, yoh_train, yoh_test = train_test_split(
    X_oh, y_oh, test_size=0.2, random_state=42,
)


## 4. Random Forest

Initialize `RandomForestClassifier(random_state=42)`, fit on the training fold, predict `X_test` into `y_pred`. Name the model `clf` so the evaluation cell matches the brief.


In [ ]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("predicted positive (p=1) count:", int(y_pred.sum()))


## 5. Model evaluation

Print accuracy, the numeric confusion matrix, and the classification report (readable names if you want). Heatmap the matrix. Horizontal bar of the **top 5** Gini importances.

On this seed the default forest is typically **perfect** on the 1,625-row test fold: accuracy 1.0, off-diagonal zeros. That is a property of *this table*, not a license to eat wild mushrooms.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy score:", accuracy)
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
print("Classification report:")
print(classification_report(
    y_test, y_pred, target_names=["edible (e=0)", "poisonous (p=1)"],
))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", square=True,
            xticklabels=["edible", "poisonous"],
            yticklabels=["edible", "poisonous"],
            cbar_kws={"shrink": 0.8})
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion matrix — Random Forest")
plt.tight_layout(); plt.show()

importances = pd.Series(clf.feature_importances_, index=X_train.columns)
top5 = importances.sort_values(ascending=False).head(5)
print(importances.sort_values(ascending=False).head(8))
plt.figure(figsize=(8, 5))
ax = sns.barplot(x=top5.values, y=top5.index, orient="h", color="#6d4aff")
ax.set_title("Top 5 feature importances — Random Forest")
ax.set_xlabel("Gini importance")
for i, v in enumerate(top5.values):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()


## 6. Alternate code that reaches the same decision

Fit a single `DecisionTreeClassifier`, a one-hot `LogisticRegression`, an odor-majority rule, and a mutual-information ranking. On this table DT and one-hot LR also hit 1.0; the odor rule sits near **0.985** because odor=`n` is mixed.


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("DT acc", accuracy_score(y_test, dt.predict(X_test)))

lr = LogisticRegression(max_iter=2000, solver="liblinear")
lr.fit(Xoh_train, yoh_train)
print("one-hot LogReg acc", accuracy_score(yoh_test, lr.predict(Xoh_test)),
      "width", X_oh.shape[1])

maj = df.groupby("odor")["class"].agg(lambda s: s.value_counts().idxmax())
odor_pred = df["odor"].map(maj)
odor_acc = float((odor_pred == df["class"]).mean())
print("odor majority-rule acc (full table)", odor_acc)
print("odor='n' mix:\n", df.loc[df["odor"] == "n", "class"].value_counts())

mi = pd.Series(
    mutual_info_classif(X, y, random_state=42), index=X.columns,
).sort_values(ascending=False)
print("\nMutual information with class:")
print(mi.head(8))


## 7. More practice

Work three small drills.

**A.** Restrict the test fold to `habitat == d` (woods) — rebuild a tiny pipeline from the raw frame, or filter the already-encoded rows if you kept a habitat column. Does accuracy stay perfect?

**B.** Cost matrix. Treat a false edible as 10× worse than a false poisonous. Using `predict_proba`, sweep a threshold that flags poisonous more aggressively. How many extra false poisonous do you buy to drive false edibles to zero?

**C.** A 2-feature card: `odor` + `spore-print-color` only. Compare test accuracy and the costly-cell count to the full 21-feature forest.


In [ ]:
# A — woods habitat on the encoded frame
woods = X_test["habitat"] == label_encoders["habitat"].transform(["d"])[0]
print("woods test rows:", int(woods.sum()),
      "acc", accuracy_score(y_test[woods], y_pred[woods]) if woods.any() else None)

# B — probability threshold
proba = clf.predict_proba(X_test)[:, 1]  # P(poisonous)
print("\nthreshold | false_edible | false_poison | acc")
for t in [0.50, 0.30, 0.10, 0.05]:
    pred_t = (proba >= t).astype(int)
    false_edible = int(((y_test == 1) & (pred_t == 0)).sum())
    false_poison = int(((y_test == 0) & (pred_t == 1)).sum())
    acc_t = accuracy_score(y_test, pred_t)
    print(f"  {t:4.2f}     | {false_edible:12d} | {false_poison:12d} | {acc_t:.4f}")

# C — odor + spore-print-color
cols2 = ["odor", "spore-print-color"]
clf2 = RandomForestClassifier(random_state=42)
clf2.fit(X_train[cols2], y_train)
yp2 = clf2.predict(X_test[cols2])
print("\n2-feature acc", accuracy_score(y_test, yp2),
      "false edibles", int(((y_test == 1) & (yp2 == 0)).sum()))


## 8. Simulation — twist a few knobs

Default RF is already perfect, so accuracy-vs-`n_estimators` is a flat line. The knobs that actually move:

| Knob | What we change | What usually happens on this table |
|------|----------------|------------------------------------|
| `max_depth` | 1 → None | depth 1 ≈ 0.87; depth 8 = 1.0 |
| drop features | remove odor / gill-size / spore-print | still ~1.0 until you keep *only* odor (~0.985) |
| label flip | flip 0–35% of *train* labels | test acc holds until ~10%, then falls |
| training n | 50, 100, …, 6,499 | 50 rows ≈ 0.94; 800 rows already ~1.0 |

Edit the four lists, re-run, read the 2×2 panel. A saved reference lives at `mushedib_simulation.png`.


In [ ]:
rng = np.random.default_rng(42)
DEPTHS = [1, 2, 3, 4, 5, 6, 8, None]
FLIP_RATES = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35]
TRAIN_NS = [50, 100, 200, 400, 800, 1600, 3200, len(X_train)]

fig, axes = plt.subplots(2, 2, figsize=(10.6, 7.8))

dacc = []
for d in DEPTHS:
    m = RandomForestClassifier(n_estimators=50, max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    dacc.append(accuracy_score(y_test, m.predict(X_test)))
labs = ["None" if d is None else str(d) for d in DEPTHS]
axes[0, 0].plot(range(len(DEPTHS)), dacc, marker="o", color="#cc4125")
axes[0, 0].set_xticks(range(len(DEPTHS))); axes[0, 0].set_xticklabels(labs)
axes[0, 0].set_title("Accuracy vs max_depth (50 trees)")
axes[0, 0].set_xlabel("max_depth"); axes[0, 0].set_ylabel("test acc")
print("depth", list(zip(labs, [round(a, 4) for a in dacc])))

drop_plan = {
    "all 21": [],
    "no odor": ["odor"],
    "no odor+gill-size": ["odor", "gill-size"],
    "no odor+spore": ["odor", "spore-print-color"],
    "only odor": None,
}
names, accs = [], []
for name, cols in drop_plan.items():
    if name == "only odor":
        Xt, Xe = X_train[["odor"]], X_test[["odor"]]
    else:
        Xt, Xe = X_train.drop(columns=cols), X_test.drop(columns=cols)
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(Xt, y_train)
    names.append(name); accs.append(accuracy_score(y_test, m.predict(Xe)))
axes[0, 1].barh(names, accs, color="#6d4aff")
axes[0, 1].set_xlim(0.85, 1.005)
axes[0, 1].set_title("Accuracy after dropping key features")
print("drop", list(zip(names, [round(a, 4) for a in accs])))

nacc, nrec = [], []
ytr_np = y_train.to_numpy()
for f in FLIP_RATES:
    y_noisy = ytr_np.copy()
    k = int(f * len(y_noisy))
    if k:
        idx = rng.choice(len(y_noisy), size=k, replace=False)
        y_noisy[idx] = 1 - y_noisy[idx]
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(X_train, y_noisy)
    yp = m.predict(X_test)
    nacc.append(accuracy_score(y_test, yp))
    nrec.append(recall_score(y_test, yp, pos_label=1))
axes[1, 0].plot([f * 100 for f in FLIP_RATES], nacc, marker="o", label="accuracy", color="#6aa84f")
axes[1, 0].plot([f * 100 for f in FLIP_RATES], nrec, marker="s", label="poison recall", color="#cc4125")
axes[1, 0].set_title("Train label-flip vs test metrics")
axes[1, 0].set_xlabel("flipped labels (%)"); axes[1, 0].legend()
print("noise acc", list(zip(FLIP_RATES, [round(a, 4) for a in nacc])))

sacc = []
perm = rng.permutation(len(X_train))
for n in TRAIN_NS:
    take = perm[:n]
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(X_train.iloc[take], y_train.iloc[take])
    sacc.append(accuracy_score(y_test, m.predict(X_test)))
axes[1, 1].plot(TRAIN_NS, sacc, marker="o", color="#e69138")
axes[1, 1].set_title("Accuracy vs random training n")
axes[1, 1].set_xlabel("training rows")
print("n-sub", list(zip(TRAIN_NS, [round(a, 4) for a in sacc])))

plt.suptitle("MushEdib simulation knobs")
plt.tight_layout(); plt.show()


## 9. Audience notes (rewrite the same result four ways)

Use the attached audience PDFs (*What to Consider When Considering the Audience*, *Audience and Situation Analysis*). Same numbers, four pitches.

| Audience | Data literacy | Subject knowledge | Time span | What to show |
|----------|---------------|-------------------|-----------|--------------|
| Expert (mycologist / ML) | high | high | long | odor split, MI vs Gini, why 1.0 is not external validity, `u` vs cup collision |
| Technician (app / field key) | medium | high practical | short | the 2-feature card, the costly cell, “do not ship as eat/don’t-eat” |
| Executive (park / food safety) | low–medium | low–medium | very short | 51.8 / 48.2 balance, 0 false edibles on hold-out, **cannot** replace a field guide |
| Nonspecialist | low | low | short | “smell is the giveaway in *this* book of drawings, not in your backyard” |

Full prose: `MushEdib_Project_Memo.docx`.



## 10. Good fit vs limitations

**Good fit**
- All-categorical field-guide codebook with a nearly balanced binary target.
- Tree ensembles and even a shallow DT — splits on odor / spore print / gill size.
- Teaching clean vs mode-impute, LabelEncoder vs one-hot, costly-error thinking.

**Limitations / anti-applications**
- 1987 North-American field-guide codes. New species, look-alikes, regional variants, and photo-only inputs are out of scope.
- `?` → `u` collides with the official cup code.
- Perfect hold-out accuracy is a dataset artifact (features were collected *to* separate edibility).
- Never a foraging decision, never a restaurant receiving spec, never a poisoning-triage tool.

Top applications of the *pattern* (categorical RF + costly FN): defect flags, fraud typology codes, wildlife ID from field marks — always with a human in the loop.

